In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, roc_curve
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('C:/Users/diyad/Desktop/HireSafe/src')

from signal_extraction.signals import SignalExtractor

print("="*80)
print("HIRESAFE: MULTI-SIGNAL NLP PIPELINE WITH ENSEMBLE CLASSIFICATION")
print("="*80)

# Load ORIGINAL data (not augmented) for fair comparison with baselines
train_df = pd.read_csv('C:/Users/diyad/Desktop/HireSafe/data/train.csv')
val_df = pd.read_csv('C:/Users/diyad/Desktop/HireSafe/data/val.csv')
test_df = pd.read_csv('C:/Users/diyad/Desktop/HireSafe/data/test.csv')

print(f"\nTrain (original): {len(train_df)} samples ({train_df['fraudulent'].mean()*100:.1f}% fraud)")
print(f"Val: {len(val_df)} samples ({val_df['fraudulent'].mean()*100:.1f}% fraud)")
print(f"Test: {len(test_df)} samples ({test_df['fraudulent'].mean()*100:.1f}% fraud)")


HIRESAFE: MULTI-SIGNAL NLP PIPELINE WITH ENSEMBLE CLASSIFICATION

Train (original): 14304 samples (4.8% fraud)
Val: 1788 samples (4.9% fraud)
Test: 1788 samples (4.8% fraud)


In [2]:
# Initialize signal extractor
extractor = SignalExtractor()

print("\nExtracting signals for all datasets...")

# Train
print("  - Training set...")
train_signals = []
for idx, row in train_df.iterrows():
    train_signals.append(extractor.extract_all_signals(row))
    if idx % 2000 == 0:
        print(f"    Processed {idx}/{len(train_df)}...")
train_signals_df = pd.DataFrame(train_signals)

# Val
print("  - Validation set...")
val_signals = []
for idx, row in val_df.iterrows():
    val_signals.append(extractor.extract_all_signals(row))
val_signals_df = pd.DataFrame(val_signals)

# Test
print("  - Test set...")
test_signals = []
for idx, row in test_df.iterrows():
    test_signals.append(extractor.extract_all_signals(row))
test_signals_df = pd.DataFrame(test_signals)

print("\n✓ Signal extraction complete!")
print(f"Extracted features: {train_signals_df.columns.tolist()}")


Extracting signals for all datasets...
  - Training set...
    Processed 0/14304...
    Processed 2000/14304...
    Processed 4000/14304...
    Processed 6000/14304...
    Processed 8000/14304...
    Processed 10000/14304...
    Processed 12000/14304...
    Processed 14000/14304...
  - Validation set...
  - Test set...

✓ Signal extraction complete!
Extracted features: ['payment_request', 'urgency', 'offplatform_contact', 'vague_company', 'salary_anomaly']


In [3]:
# Load model
print("\nLoading Sentence-Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Scam templates
scam_templates = [
    "Urgent work from home opportunity. Send payment for training materials.",
    "Make thousands per week working remotely. Pay processing fee to start immediately.",
    "No experience needed. Contact via WhatsApp. Wire transfer required for equipment.",
    "Limited time offer. Flexible schedule. Pay upfront fee for background check.",
    "Earn $5000 weekly. Text personal number. Payment required before starting.",
    "Immediate start. High salary. Send deposit via PayPal or Western Union.",
    "Easy online work. No qualifications required. Transfer fee to secure position.",
    "Remote data entry job. Urgent hiring. Pay application fee to apply today.",
    "Work when you want. Great pay. Send money order for training program.",
    "Home-based position. Start now. Payment needed for starter kit and materials."
]

print("Encoding scam templates...")
template_embeddings = model.encode(scam_templates)

# Encode datasets
print("\nEncoding datasets (this takes 2-3 minutes on CPU)...")

print("  - Training set...")
train_embeddings = model.encode(train_df['full_text'].fillna('').tolist(), 
                                batch_size=32, show_progress_bar=True)
train_similarities = cosine_similarity(train_embeddings, template_embeddings)
train_max_sim = train_similarities.max(axis=1)

print("  - Validation set...")
val_embeddings = model.encode(val_df['full_text'].fillna('').tolist(), 
                              batch_size=32, show_progress_bar=True)
val_similarities = cosine_similarity(val_embeddings, template_embeddings)
val_max_sim = val_similarities.max(axis=1)

print("  - Test set...")
test_embeddings = model.encode(test_df['full_text'].fillna('').tolist(), 
                               batch_size=32, show_progress_bar=True)
test_similarities = cosine_similarity(test_embeddings, template_embeddings)
test_max_sim = test_similarities.max(axis=1)

print("\n✓ Semantic similarity scores computed!")


Loading Sentence-Transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding scam templates...

Encoding datasets (this takes 2-3 minutes on CPU)...
  - Training set...


Batches:   0%|          | 0/447 [00:00<?, ?it/s]

  - Validation set...


Batches:   0%|          | 0/56 [00:00<?, ?it/s]

  - Test set...


Batches:   0%|          | 0/56 [00:00<?, ?it/s]


✓ Semantic similarity scores computed!


In [4]:
# Combine signals + similarity into feature matrix
print("\nCreating feature matrices...")

# Add similarity as a feature
train_signals_df['semantic_similarity'] = train_max_sim
val_signals_df['semantic_similarity'] = val_max_sim
test_signals_df['semantic_similarity'] = test_max_sim

# Feature matrices
X_train = train_signals_df.values
X_val = val_signals_df.values
X_test = test_signals_df.values

y_train = train_df['fraudulent'].values
y_val = val_df['fraudulent'].values
y_test = test_df['fraudulent'].values

print(f"✓ Feature matrices created!")
print(f"\nFeature matrix shape: {X_train.shape}")
print(f"Features ({X_train.shape[1]}): {train_signals_df.columns.tolist()}")

print(f"\nTarget distribution:")
print(f"  Train: {y_train.mean()*100:.1f}% fraudulent")
print(f"  Val: {y_val.mean()*100:.1f}% fraudulent")
print(f"  Test: {y_test.mean()*100:.1f}% fraudulent")


Creating feature matrices...
✓ Feature matrices created!

Feature matrix shape: (14304, 6)
Features (6): ['payment_request', 'urgency', 'offplatform_contact', 'vague_company', 'salary_anomaly', 'semantic_similarity']

Target distribution:
  Train: 4.8% fraudulent
  Val: 4.9% fraudulent
  Test: 4.8% fraudulent


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

print("\n" + "="*80)
print("COMBINING SIGNAL FEATURES + TF-IDF FEATURES")
print("="*80)

# Create TF-IDF features
print("\nCreating TF-IDF features...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)

X_train_tfidf = tfidf.fit_transform(train_df['full_text'].fillna(''))
X_val_tfidf = tfidf.transform(val_df['full_text'].fillna(''))
X_test_tfidf = tfidf.transform(test_df['full_text'].fillna(''))

print(f"TF-IDF shape: {X_train_tfidf.shape}")

# Convert signal features to sparse format
X_train_signals_sparse = csr_matrix(X_train)
X_val_signals_sparse = csr_matrix(X_val)
X_test_signals_sparse = csr_matrix(X_test)

# Combine TF-IDF + Signal features
X_train_combined = hstack([X_train_tfidf, X_train_signals_sparse])
X_val_combined = hstack([X_val_tfidf, X_val_signals_sparse])
X_test_combined = hstack([X_test_tfidf, X_test_signals_sparse])

print(f"\nCombined feature matrix shape: {X_train_combined.shape}")
print(f"  - TF-IDF features: {X_train_tfidf.shape[1]}")
print(f"  - Signal features: {X_train.shape[1]}")
print(f"  - Total features: {X_train_combined.shape[1]}")

# Update feature matrices
X_train = X_train_combined
X_val = X_val_combined
X_test = X_test_combined

print("\n✓ Combined features ready!")


COMBINING SIGNAL FEATURES + TF-IDF FEATURES

Creating TF-IDF features...
TF-IDF shape: (14304, 5000)

Combined feature matrix shape: (14304, 5006)
  - TF-IDF features: 5000
  - Signal features: 6
  - Total features: 5006

✓ Combined features ready!


In [6]:
print("\n" + "="*80)
print("TRAINING LOGISTIC REGRESSION")
print("="*80)

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)

# Predict probabilities
lr_train_proba = lr_model.predict_proba(X_train)[:, 1]
lr_val_proba = lr_model.predict_proba(X_val)[:, 1]
lr_test_proba = lr_model.predict_proba(X_test)[:, 1]

print("✓ Logistic Regression trained!")


TRAINING LOGISTIC REGRESSION
✓ Logistic Regression trained!


In [10]:
print("\n" + "="*80)
print("INDIVIDUAL MODEL PERFORMANCE")
print("="*80)

# Logistic Regression alone
lr_test_pred = (lr_test_proba >= 0.5).astype(int)
lr_precision, lr_recall, lr_f1, _ = precision_recall_fscore_support(y_test, lr_test_pred, average='binary')

print("\nLogistic Regression (with signals):")
print(f"  Precision: {lr_precision:.3f}")
print(f"  Recall: {lr_recall:.3f}")
print(f"  F1-Score: {lr_f1:.3f}")

# XGBoost alone
xgb_test_pred = (xgb_test_proba >= 0.5).astype(int)
xgb_precision, xgb_recall, xgb_f1, _ = precision_recall_fscore_support(y_test, xgb_test_pred, average='binary')

print("\nXGBoost (with signals):")
print(f"  Precision: {xgb_precision:.3f}")
print(f"  Recall: {xgb_recall:.3f}")
print(f"  F1-Score: {xgb_f1:.3f}")

print("\n" + "-"*80)
print(f"Best individual model: {'XGBoost' if xgb_f1 > lr_f1 else 'Logistic Regression'}")
print(f"Best F1: {max(xgb_f1, lr_f1):.3f}")


INDIVIDUAL MODEL PERFORMANCE

Logistic Regression (with signals):
  Precision: 0.534
  Recall: 0.919
  F1-Score: 0.675

XGBoost (with signals):
  Precision: 0.641
  Recall: 0.872
  F1-Score: 0.739

--------------------------------------------------------------------------------
Best individual model: XGBoost
Best F1: 0.739


In [7]:
print("\n" + "="*80)
print("TRAINING XGBOOST")
print("="*80)

scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
print(f"Scale pos weight: {scale_pos_weight:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    learning_rate=0.1,
    max_depth=5
)
xgb_model.fit(X_train, y_train)

# Predict probabilities
xgb_train_proba = xgb_model.predict_proba(X_train)[:, 1]
xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]

print("✓ XGBoost trained!")


TRAINING XGBOOST
Scale pos weight: 19.64
✓ XGBoost trained!


In [8]:
print("\n" + "="*80)
print("CREATING ENSEMBLE (Average of LR + XGBoost)")
print("="*80)

# Average probabilities from both models
ensemble_train_proba = (lr_train_proba + xgb_train_proba) / 2
ensemble_val_proba = (lr_val_proba + xgb_val_proba) / 2
ensemble_test_proba = (lr_test_proba + xgb_test_proba) / 2

# Convert to predictions (threshold = 0.5)
ensemble_test_pred = (ensemble_test_proba >= 0.5).astype(int)

print("✓ Ensemble created!")


CREATING ENSEMBLE (Average of LR + XGBoost)
✓ Ensemble created!


In [9]:
print("\n" + "="*80)
print("HIRESAFE FINAL EVALUATION")
print("="*80)

# Test set performance
print("\nTest Set Performance:")
print(classification_report(y_test, ensemble_test_pred, 
                          target_names=['Legitimate', 'Fraudulent']))

cm_hiresafe = confusion_matrix(y_test, ensemble_test_pred)
print("\nConfusion Matrix:")
print(cm_hiresafe)

# Metrics
precision_hs, recall_hs, f1_hs, _ = precision_recall_fscore_support(
    y_test, ensemble_test_pred, average='binary'
)
auc_hs = roc_auc_score(y_test, ensemble_test_proba)

print(f"\n📊 HIRESAFE SUMMARY METRICS:")
print(f"Precision: {precision_hs:.3f}")
print(f"Recall: {recall_hs:.3f}")
print(f"F1-Score: {f1_hs:.3f}")
print(f"AUC-ROC: {auc_hs:.3f}")

print(f"\nConfusion Matrix Breakdown:")
print(f"True Negatives: {cm_hiresafe[0,0]}")
print(f"False Positives: {cm_hiresafe[0,1]}")
print(f"False Negatives: {cm_hiresafe[1,0]}")
print(f"True Positives: {cm_hiresafe[1,1]}")

# Compare to best baseline
print("\n" + "="*80)
print("COMPARISON TO BEST BASELINE (TF-IDF + XGBoost)")
print("="*80)
baseline_f1 = 0.864
print(f"Baseline F1:  {baseline_f1:.3f}")
print(f"HireSafe F1:  {f1_hs:.3f}")
print(f"Improvement:  {'+' if f1_hs > baseline_f1 else ''}{(f1_hs - baseline_f1):.3f}")

if f1_hs >= baseline_f1 + 0.05:
    print("\n🎉 HYPOTHESIS CONFIRMED! HireSafe beats baseline by ≥5%!")
elif f1_hs > baseline_f1:
    print("\n✓ HireSafe beats baseline (but <5% improvement)")
else:
    print("\n⚠️ HireSafe does not beat baseline")



HIRESAFE FINAL EVALUATION

Test Set Performance:
              precision    recall  f1-score   support

  Legitimate       1.00      0.97      0.98      1702
  Fraudulent       0.63      0.92      0.75        86

    accuracy                           0.97      1788
   macro avg       0.81      0.95      0.87      1788
weighted avg       0.98      0.97      0.97      1788


Confusion Matrix:
[[1656   46]
 [   7   79]]

📊 HIRESAFE SUMMARY METRICS:
Precision: 0.632
Recall: 0.919
F1-Score: 0.749
AUC-ROC: 0.993

Confusion Matrix Breakdown:
True Negatives: 1656
False Positives: 46
False Negatives: 7
True Positives: 79

COMPARISON TO BEST BASELINE (TF-IDF + XGBoost)
Baseline F1:  0.864
HireSafe F1:  0.749
Improvement:  -0.115

⚠️ HireSafe does not beat baseline


In [13]:
from sklearn.model_selection import cross_val_score
import numpy as np
from scipy import stats

print("="*80)
print("STATISTICAL SIGNIFICANCE TESTING")
print("="*80)

# We'll compare TF-IDF + XGBoost vs TF-IDF + Logistic Regression
# using 5-fold cross-validation

# Prepare original training data with TF-IDF only
train_original = pd.read_csv('C:/Users/diyad/Desktop/HireSafe/data/train.csv')
tfidf_cv = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tfidf_cv = tfidf_cv.fit_transform(train_original['full_text'].fillna(''))
y_train_cv = train_original['fraudulent'].values

print("\nRunning 5-fold cross-validation...")
print("This will take a few minutes...\n")

# Logistic Regression CV
print("Evaluating Logistic Regression...")
lr_cv = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_scores = cross_val_score(lr_cv, X_train_tfidf_cv, y_train_cv, cv=5, scoring='f1', n_jobs=-1)

print(f"LR F1 scores: {lr_scores}")
print(f"LR F1: {lr_scores.mean():.3f} ± {lr_scores.std():.3f}")

# XGBoost CV
print("\nEvaluating XGBoost...")
scale_pos = len(y_train_cv[y_train_cv==0]) / len(y_train_cv[y_train_cv==1])
xgb_cv = xgb.XGBClassifier(n_estimators=100, scale_pos_weight=scale_pos, random_state=42)
xgb_scores = cross_val_score(xgb_cv, X_train_tfidf_cv, y_train_cv, cv=5, scoring='f1', n_jobs=-1)

print(f"XGB F1 scores: {xgb_scores}")
print(f"XGB F1: {xgb_scores.mean():.3f} ± {xgb_scores.std():.3f}")

# Paired t-test
t_stat, p_value = stats.ttest_rel(xgb_scores, lr_scores)

print("\n" + "="*80)
print("STATISTICAL TEST RESULTS")
print("="*80)

print(f"\nLogistic Regression: {lr_scores.mean():.3f} ± {lr_scores.std():.3f}")
print(f"XGBoost:            {xgb_scores.mean():.3f} ± {xgb_scores.std():.3f}")
print(f"\nPaired t-test:")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value: {p_value:.4f}")

if p_value < 0.05:
    print(f"\n✓ XGBoost significantly outperforms Logistic Regression (p < 0.05)")
else:
    print(f"\n  No significant difference (p ≥ 0.05)")

print("\n📝 For your thesis:")
print(f'   "XGBoost achieved an F1-score of {xgb_scores.mean():.3f} ± {xgb_scores.std():.3f}')
print(f'    across 5-fold cross-validation, significantly outperforming')
print(f'    Logistic Regression ({lr_scores.mean():.3f} ± {lr_scores.std():.3f}, p = {p_value:.4f})."')

STATISTICAL SIGNIFICANCE TESTING

Running 5-fold cross-validation...
This will take a few minutes...

Evaluating Logistic Regression...
LR F1 scores: [0.70056497 0.68023256 0.64942529 0.70447761 0.67422096]
LR F1: 0.682 ± 0.020

Evaluating XGBoost...
XGB F1 scores: [0.80866426 0.79681275 0.74615385 0.82307692 0.79365079]
XGB F1: 0.794 ± 0.026

STATISTICAL TEST RESULTS

Logistic Regression: 0.682 ± 0.020
XGBoost:            0.794 ± 0.026

Paired t-test:
  t-statistic: 26.080
  p-value: 0.0000

✓ XGBoost significantly outperforms Logistic Regression (p < 0.05)

📝 For your thesis:
   "XGBoost achieved an F1-score of 0.794 ± 0.026
    across 5-fold cross-validation, significantly outperforming
    Logistic Regression (0.682 ± 0.020, p = 0.0000)."
